### ColBERT

之前的技术实在研究如何更好地组织文档，那么ColBERT研究地是如何更精细地表示文档

#### 一、核心动机：拒绝过度压缩
在基础的RAG中，使用双编码器
- 传统做法：嵌入模型接受一个文档，将其压缩成一个单一的固定长度
- 痛点：这种做法就像把一本书的内容强行压缩成一个词。虽然处理速度会块，但会丢很多细节和上下文信息


#### 二、ColBERT：多向量表示与晚期交互
1. 词级编码 (Token-level Embedding)： ColBERT 不会为整个文档生成一个向量，而是为文档中的每一个 Token（词块）都生成一个对应的向量。同时，它在生成向量时会考虑词的位置权重
2. 问题的处理： 同样地，它也会为用户提出的问题中的每一个 Token 生成独立的向量
3. 最大相似度匹配 (MaxSim)： 这是 ColBERT 的灵魂算法。检索时，系统会拿问题中的每一个词去和文档中的每一个词做对比：
    ◦ 对于问题中的 Token 1，找到文档中与之最相似的那个 Token，取其最大相似度得分
    ◦ 对问题中所有的 Token 都重复此操作
    ◦ 最终得分： 将所有这些“最大相似度得分”相加，作为该文档与问题的匹配分数　　
    
优势： 由于保留了每个词的特征，ColBERT 在处理长文档和复杂语义匹配时，表现得非常强劲且细腻

#### 三、性能与生产环境建议
- 延迟问题： 由于需要计算“词对词”的相似度，ColBERT 的计算开销比传统的单一向量搜索要大。在考虑大规模投入生产环境时，需要关注其检索延迟
- 适用场景： 如果你的 RAG 应用需要处理长文档，或者对语义精度有极高的要求，ColBERT 是一个非常值得尝试的高级方案

In [1]:
import requests
from pylate import models, indexes, retrieve
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from typing import List
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_models.tongyi import ChatTongyi
EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # 免费嵌入模型
model = models.ColBERT(
    model_name_or_path="lightonai/GTE-ModernColBERT-v1",
)

c:\Users\23017\anaconda3\envs\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:

#  获取维基百科页面文本
def get_wikipedia_page(title: str):
    """
    获取维基百科页面的完整文本内容。

    :param title: str - 维基百科页面标题
    :return: str - 页面文本内容，若不存在则返回 None
    """
    URL = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }
    headers = {"User-Agent": "RAGatouille_tutorial/0.0.1 (ben@clavie.eu)"}
    response = requests.get(URL, params=params, headers=headers)
    data = response.json()
    page = next(iter(data["query"]["pages"].values()))
    return page["extract"] if "extract" in page else None

full_document = get_wikipedia_page("Hayao_Miyazaki")
print(full_document)

chunk_size = 180
chunks = [full_document[i:i+chunk_size] for i in range(0, len(full_document), chunk_size)]

# 生成块 ID
chunk_ids = [f"doc_{i}" for i in range(len(chunks))]
print(chunk_ids)
# 建立 ID → 文本的映射
id_to_text = dict(zip(chunk_ids, chunks))
print(id_to_text)


Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves as its honorary chairman. Throughout his career, Miyazaki has attained international acclaim as a masterful storyteller and creator of Japanese animated feature films, and is widely regarded as one of the most accomplished filmmakers in the history of animation.
Born in Tokyo City, Miyazaki expressed interest in manga and animation from an early age. He joined Toei Animation in 1963, working as an inbetween artist and key animator on films like Gulliver's Travels Beyond the Moon (1965), Puss in Boots (1969), and Animal Treasure Island (1971), before moving to A-Pro in 1971, where he co-directed Lupin the Third Part I (1971–1972) alongside Isao Takahata. After moving to Zuiyō Eizō (later Nippon Animation) in 1973, Miyazaki worked as an animator on World Masterpiece Theater and directed the television series Fut

In [17]:

# 初始化索引
index = indexes.PLAID(
    collection = chunks,
    index_name = "Miyazaki-123",
    max_document_length=180,
    split_documents=False, 
)

documents_embeddings = model.encode(
    sentences=full_document,
    batch_size=64,
    is_query=False,
    show_progress_bar=True,
)

index.add_documents(
    documents_ids=chunk_ids,
    documents_embeddings=documents_embeddings,
)

Encoding documents (bs=64): 100%|██████████| 1/1 [00:00<00:00,  4.11it/s]


In [19]:
from typing import List, Any ,Dict
from pydantic import Field
retriever = retrieve.ColBERT(index=index)
class PylateRetriever(BaseRetriever):
    retriever: Any = Field(description="The underlying pylate retriever")
    model: Any = Field(description="The ColBERT model used for encoding queries")
    k: int = Field(default=3, description="Number of documents to retrieve")
    id_to_text: Dict[str, str] = Field(default_factory=dict, description="Mapping from doc_id to text")

    def _get_relevant_documents(self, query: str, **kwargs) -> List[Document]:
        # 编码查询
        query_embedding = self.model.encode([query], is_query=True, show_progress_bar=False)

        # 执行检索
        raw_results = self.retriever.retrieve(queries_embeddings=query_embedding, k=self.k)
        results = raw_results[0] if (raw_results and isinstance(raw_results[0], list)) else raw_results

        # 将 ID 转换为文本
        docs = []
        for res in results:
            doc_id = res["id"]                     # 假设返回的字典包含 "id" 和 "score"
            text = self.id_to_text.get(doc_id, f"[Missing text for {doc_id}]")
            docs.append(Document(page_content=text, metadata={"score": res["score"], "doc_id": doc_id}))
        return docs

    async def _aget_relevant_documents(self, query: str, **kwargs) -> List[Document]:
        return self._get_relevant_documents(query, **kwargs)

In [ ]:
langchain_retriever = PylateRetriever(
    retriever=retriever,
    model=model,
    k=3,
    id_to_text=id_to_text      # 传入映射
)

query = "What animation studio did Miyazaki found?"
lc_results = langchain_retriever.invoke(query)

print("\n✅ 检索结果（含真实文本）：")
for i, doc in enumerate(lc_results):
    print(f"{i+1}. 分数: {doc.metadata['score']:.4f}")
    print(f"   文本片段: {doc.page_content[:200]}...\n")

[Document(metadata={'score': 11.3896484375}, page_content='doc_0'), Document(metadata={'score': 11.3896484375}, page_content='doc_0')]


In [23]:
langchain_retriever = PylateRetriever(
    retriever=retriever,
    model=model,
    k=3,
    id_to_text=id_to_text      # 传入映射
)

query = "What animation studio did Miyazaki found?"
lc_results = langchain_retriever.invoke(query)

print("\n检索结果（含真实文本）：")
for i, doc in enumerate(lc_results):
    print(f"{i+1}. 分数: {doc.metadata['score']:.4f}")
    print(f"   文本片段: {doc.page_content[:200]}...\n")


检索结果（含真实文本）：
1. 分数: 11.3896
   文本片段: Hayao Miyazaki (宮崎 駿 or 宮﨑 駿, Miyazaki Hayao; [mijaꜜzaki hajao]; born January 5, 1941) is a Japanese animator, filmmaker, and manga artist. He co-founded Studio Ghibli and serves a...

